In [2]:
import yfinance as yf

# Load the same historical QQQ data
qqq = yf.Ticker("QQQ").history(
    start="2010-01-01",
    end="2026-09-02",
    interval="1d",
    auto_adjust=True,
    actions=False
)

# Recalculate daily returns
qqq["Daily_Return_Pct"] = qqq["Close"].pct_change() * 100

# Verify the data
qqq.shape

(4191, 6)

In [3]:
# Create the next-trading-day return target
qqq["Target_Next_Day_Return_Pct"] = qqq["Daily_Return_Pct"].shift(-1)

# Compare today's return with tomorrow's return
qqq[["Daily_Return_Pct", "Target_Next_Day_Return_Pct"]].head(10)


,Daily_Return_Pct,Target_Next_Day_Return_Pct
Date,,
2010-01-04 00:00:00-05:00,NaN,0.000000
2010-01-05 00:00:00-05:00,0.000000,-0.603165
2010-01-06 00:00:00-05:00,-0.603165,0.064969
2010-01-07 00:00:00-05:00,0.064969,0.823036
2010-01-08 00:00:00-05:00,0.823036,-0.408112
2010-01-11 00:00:00-05:00,-0.408112,-1.251070
2010-01-12 00:00:00-05:00,-1.251070,1.245100
2010-01-13 00:00:00-05:00,1.245100,0.086283
2010-01-14 00:00:00-05:00,0.086283,-1.164052


In [ ]:
# Calculate 5-day momentum
qqq["Momentum_5d_Pct"] = qqq["Close"].pct_change(periods=5) * 100

# Display our new feature
qqq[["Close", "Daily_Return_Pct", "Momentum_5d_Pct"]].head(10)

,Close,Daily_Return_Pct,Momentum_5d_Pct
Date,,,
2010-01-04 00:00:00-05:00,40.204662,NaN,NaN
2010-01-05 00:00:00-05:00,40.204662,0.000000,NaN
2010-01-06 00:00:00-05:00,39.962162,-0.603165,NaN
2010-01-07 00:00:00-05:00,39.988125,0.064969,NaN
2010-01-08 00:00:00-05:00,40.317242,0.823036,NaN
2010-01-11 00:00:00-05:00,40.152702,-0.408112,-0.129239
2010-01-12 00:00:00-05:00,39.650364,-1.251070,-1.378692
2010-01-13 00:00:00-05:00,40.144051,1.245100,0.455152
2010-01-14 00:00:00-05:00,40.178688,0.086283,0.476549


In [7]:
# Calculate 20-day rolling volatility
qqq["Volatility_20d_Pct"] = qqq["Daily_Return_Pct"].rolling(window=20).std()

# Examine the first valid volatility observations
qqq[["Daily_Return_Pct", "Volatility_20d_Pct"]].iloc[19:25]

,Daily_Return_Pct,Volatility_20d_Pct
Date,,
2010-02-01 00:00:00-05:00,1.098388,NaN
2010-02-02 00:00:00-05:00,0.901552,1.265401
2010-02-03 00:00:00-05:00,0.549851,1.278160
2010-02-04 00:00:00-05:00,-2.893640,1.405649
2010-02-05 00:00:00-05:00,0.844653,1.429440
2010-02-08 00:00:00-05:00,-0.721268,1.404370


In [ ]:

# Calculate the 20-day simple moving average
qqq["SMA_20d"] = qqq["Close"].rolling(window=20).mean()

# Calculate percentage distance from the moving average
qqq["MA_Distance_20d_Pct"] = (
    (qqq["Close"] / qqq["SMA_20d"]) - 1
) * 100

# Display the new feature
qqq[["Close", "SMA_20d", "MA_Distance_20d_Pct"]].iloc[18:24]

,Close,SMA_20d,MA_Distance_20d_Pct
Date,,,
2010-01-29 00:00:00-05:00,37.060692,NaN,NaN
2010-02-01 00:00:00-05:00,37.467762,39.301749,-4.666427
2010-02-02 00:00:00-05:00,37.805553,39.181794,-3.512449
2010-02-03 00:00:00-05:00,38.013428,39.072232,-2.709864
2010-02-04 00:00:00-05:00,36.913456,38.919797,-5.155065
2010-02-05 00:00:00-05:00,37.225246,38.781653,-4.013255


In [10]:
# Define our four predictive features
feature_cols = [
    "Daily_Return_Pct",
    "Momentum_5d_Pct",
    "Volatility_20d_Pct",
    "MA_Distance_20d_Pct"
]

# Define the prediction target
target_col = "Target_Next_Day_Return_Pct"

# Create a dataset containing only our features and target
model_data = qqq[feature_cols + [target_col]].copy()

# Check the dataset before cleaning
print("Rows before cleaning:", len(model_data))
print("\nMissing values by column:")
print(model_data.isna().sum())


Rows before cleaning: 4191

Missing values by column:
Daily_Return_Pct               1
Momentum_5d_Pct                5
Volatility_20d_Pct            20
MA_Distance_20d_Pct           19
Target_Next_Day_Return_Pct     1
dtype: int64


In [11]:
# Remove rows with missing features or targets
model_data = model_data.dropna().copy()

# Verify the cleaned dataset
print("Rows after cleaning:", len(model_data))
print("Missing values remaining:", model_data.isna().sum().sum())

# Check the remaining date range
print("First date:", model_data.index.min().date())
print("Last date:", model_data.index.max().date())

# Display the first five observations
model_data.head()


Rows after cleaning: 4170
Missing values remaining: 0
First date: 2010-02-02
Last date: 2026-08-31


,Daily_Return_Pct,Momentum_5d_Pct,Volatility_20d_Pct,MA_Distance_20d_Pct,Target_Next_Day_Return_Pct
Date,,,,,
2010-02-02 00:00:00-05:00,0.901552,-1.578332,1.265401,-3.512449,0.549851
2010-02-03 00:00:00-05:00,0.549851,-1.812001,1.278160,-2.709864,-2.893640
2010-02-04 00:00:00-05:00,-2.893640,-2.135501,1.405649,-5.155065,0.844653
2010-02-05 00:00:00-05:00,0.844653,0.444014,1.429440,-4.013255,-0.721268
2010-02-08 00:00:00-05:00,-0.721268,-1.363864,1.404370,-4.290909,1.031256


In [12]:
from pathlib import Path

# Locate the main project folder
project_dir = Path.cwd()
if project_dir.name == "notebooks":
    project_dir = project_dir.parent

# Create a data folder if it doesn't exist
data_dir = project_dir / "data"
data_dir.mkdir(exist_ok=True)

# Prepare the dataset for saving
export_data = model_data.copy()
export_data.index = export_data.index.strftime("%Y-%m-%d")

# Save the completed dataset
csv_path = data_dir / "qqq_model_data.csv"
export_data.to_csv(csv_path, index_label="Date")

print("Saved to:", csv_path)
print("Rows saved:", len(export_data))


Saved to: /Users/michaelsipahimalani/Projects/qqq-returns-ml/data/qqq_model_data.csv
Rows saved: 4170
